# Advanced Python Class Decorators — Problems with Complete Solutions

This notebook is an expanded, practice-heavy problem set on **class decorators**.

It follows the supplied material's main progression and extends it with advanced design exercises:

- mutating classes with decorators,
- parameterized class decorators,
- injecting attributes and methods,
- registries and class contracts,
- whole-class method decoration,
- `staticmethod` and `classmethod` descriptor handling,
- `property` getter/setter/deleter handling,
- safer inspection with `inspect`,
- opt-out markers and idempotence,
- inheritance semantics,
- timing, call statistics, validation, ordering, and caching,
- a production-style reusable member transformer.

## Best practices used throughout

1. Return the decorated class unless replacement is intentional.
2. Validate decorator arguments early.
3. Use `functools.wraps` for wrapped functions.
4. Inspect `vars(cls)` / `cls.__dict__` for raw descriptors.
5. Treat `staticmethod`, `classmethod`, and `property` explicitly.
6. Avoid blindly wrapping every `callable(...)`.
7. Preserve descriptors when replacing their underlying functions.
8. Make repeated decoration idempotent when practical.
9. Fail early with clear errors for invalid classes.
10. Verify behavior with assertions.

## How to use this notebook

For each problem, try the prompt first, then compare with the complete solution and run its verification cell. All solutions use only the Python standard library.

In [1]:
from __future__ import annotations

from functools import wraps, total_ordering
import inspect
import time
from typing import Any, Callable

print("Environment ready.")

Environment ready.


---
## Problem 1 — Inject class metadata

Write a class decorator `entity` that adds `kind = "entity"` and `decorated = True` to a class. Mutate and return the original class.

In [2]:
def entity(cls):
    cls.kind = "entity"
    cls.decorated = True
    return cls


@entity
class Customer:
    def __init__(self, name):
        self.name = name

    def label(self):
        return f"Customer<{self.name}>"

In [3]:
c = Customer("Ada")
assert Customer.kind == "entity"
assert Customer.decorated is True
assert c.label() == "Customer<Ada>"
print(Customer.kind, Customer.decorated, c.label())

entity True Customer<Ada>


---
## Problem 2 — Parameterized metadata decorator

Create `@tagged(kind="service", version=3)`. Validate that `kind` is a non-empty string and `version` is a positive integer.

In [4]:
def tagged(*, kind: str, version: int):
    if not isinstance(kind, str) or not kind.strip():
        raise ValueError("kind must be a non-empty string")
    if not isinstance(version, int) or isinstance(version, bool) or version <= 0:
        raise ValueError("version must be a positive integer")

    def decorate(cls):
        cls.kind = kind.strip()
        cls.version = version
        return cls

    return decorate


@tagged(kind="service", version=3)
class BillingService:
    pass

In [5]:
assert BillingService.kind == "service"
assert BillingService.version == 3

for bad_args in ({"kind": "", "version": 1}, {"kind": "x", "version": 0}):
    try:
        tagged(**bad_args)
    except ValueError as exc:
        print(type(exc).__name__, exc)

ValueError kind must be a non-empty string
ValueError version must be a positive integer


---
## Problem 3 — Safely inject a method

Write `@inject_summary` that adds `summary(self)` and refuses to overwrite an existing `summary` member.

In [6]:
def inject_summary(cls):
    if "summary" in vars(cls):
        raise TypeError(f"{cls.__name__} already defines summary")

    def summary(self):
        parts = ", ".join(
            f"{name}={value!r}" for name, value in sorted(vars(self).items())
        )
        return f"{type(self).__name__}({parts})"

    cls.summary = summary
    return cls


@inject_summary
class Product:
    def __init__(self, sku, price):
        self.sku = sku
        self.price = price

In [7]:
p = Product("P-100", 12.5)
assert p.summary() == "Product(price=12.5, sku='P-100')"
print(p.summary())

Product(price=12.5, sku='P-100')


---
## Problem 4 — Build a class registry

Create `@register(name)` that stores classes in a registry and rejects duplicate keys. Then instantiate a class through the registry.

In [8]:
CLASS_REGISTRY = {}


def register(name: str):
    if not isinstance(name, str) or not name.strip():
        raise ValueError("registry name must be a non-empty string")
    key = name.strip()

    def decorate(cls):
        if key in CLASS_REGISTRY:
            raise KeyError(f"duplicate registry key: {key!r}")
        CLASS_REGISTRY[key] = cls
        cls.registry_name = key
        return cls

    return decorate


@register("json")
class JsonRenderer:
    def render(self, data):
        return f"json:{data!r}"


@register("text")
class TextRenderer:
    def render(self, data):
        return str(data)

In [9]:
renderer = CLASS_REGISTRY["json"]()
assert renderer.render({"x": 1}) == "json:{'x': 1}"
assert JsonRenderer.registry_name == "json"
print(CLASS_REGISTRY)

{'json': <class '__main__.JsonRenderer'>, 'text': <class '__main__.TextRenderer'>}


---
## Problem 5 — Enforce a structural class contract

Implement `@requires_methods("start", "stop")`. Use `inspect.getattr_static` so validation does not accidentally trigger descriptors.

In [10]:
def requires_methods(*method_names: str):
    if not method_names:
        raise ValueError("at least one method name is required")

    def decorate(cls):
        missing, non_callable = [], []
        for name in method_names:
            try:
                raw = inspect.getattr_static(cls, name)
            except AttributeError:
                missing.append(name)
                continue

            target = raw.__func__ if isinstance(raw, (staticmethod, classmethod)) else raw
            if not callable(target):
                non_callable.append(name)

        if missing or non_callable:
            parts = []
            if missing:
                parts.append(f"missing={missing}")
            if non_callable:
                parts.append(f"non_callable={non_callable}")
            raise TypeError(f"{cls.__name__} violates method contract: " + ", ".join(parts))
        return cls

    return decorate


@requires_methods("start", "stop")
class Worker:
    def start(self):
        return "started"

    def stop(self):
        return "stopped"

In [11]:
w = Worker()
assert w.start() == "started" and w.stop() == "stopped"

try:
    @requires_methods("run", "cancel")
    class BrokenWorker:
        run = 123
except TypeError as exc:
    print(type(exc).__name__, exc)

TypeError BrokenWorker violates method contract: missing=['cancel'], non_callable=['run']


---
## Problem 6 — Class-wide logging of ordinary instance methods

Create `log_call(fn)` and `@log_instance_methods`. Wrap only ordinary functions found directly in the class namespace. A nested class or callable object must not be replaced.

In [12]:
def log_call(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"CALL {fn.__qualname__} args={args!r} kwargs={kwargs!r} -> {result!r}")
        return result
    wrapper.__logged__ = True
    return wrapper


def log_instance_methods(cls):
    for name, obj in list(vars(cls).items()):
        if inspect.isfunction(obj):
            setattr(cls, name, log_call(obj))
    return cls


class CallableHelper:
    def __call__(self):
        return "helper called"


@log_instance_methods
class Calculator:
    helper = CallableHelper()

    def add(self, a, b):
        return a + b

    def mul(self, a, b):
        return a * b

In [13]:
calc = Calculator()
assert calc.add(2, 3) == 5
assert calc.mul(4, 5) == 20
assert isinstance(Calculator.helper, CallableHelper)
assert Calculator.helper() == "helper called"
print("Callable helper preserved:", Calculator.helper())

CALL Calculator.add args=(<__main__.Calculator object at 0x0000023F2D0CECF0>, 2, 3) kwargs={} -> 5
CALL Calculator.mul args=(<__main__.Calculator object at 0x0000023F2D0CECF0>, 4, 5) kwargs={} -> 20
Callable helper preserved: helper called


---
## Problem 7 — Inspect descriptor behavior

For instance, static, and class methods, compare the raw object in `vars(cls)` with normal `getattr(cls, name)`. Observe that static/class methods are stored as descriptors in the class namespace.

In [14]:
class DescriptorDemo:
    def instance_method(self):
        return "instance"

    @staticmethod
    def static_method():
        return "static"

    @classmethod
    def class_method(cls):
        return cls.__name__


for name in ("instance_method", "static_method", "class_method"):
    raw = vars(DescriptorDemo)[name]
    accessed = getattr(DescriptorDemo, name)
    print(
        f"{name:16} raw_type={type(raw).__name__:12} "
        f"raw_callable={callable(raw)!s:5} accessed_type={type(accessed).__name__}"
    )

instance_method  raw_type=function     raw_callable=True  accessed_type=function
static_method    raw_type=staticmethod raw_callable=True  accessed_type=function
class_method     raw_type=classmethod  raw_callable=False accessed_type=method


### Solution discussion

`vars(cls)` exposes the raw namespace. A normal method is stored as a function, while static and class methods are stored as `staticmethod` / `classmethod` descriptor objects. Normal attribute access applies descriptor binding rules, so a class decorator that needs to preserve descriptor type should inspect raw namespace entries.

---
## Problem 8 — Decorate instance, static, and class methods correctly

Implement `@log_methods`. For static/class methods, extract `.__func__`, decorate it, then re-wrap with the same descriptor type.

In [15]:
def log_methods(cls):
    for name, obj in list(vars(cls).items()):
        if isinstance(obj, staticmethod):
            setattr(cls, name, staticmethod(log_call(obj.__func__)))
        elif isinstance(obj, classmethod):
            setattr(cls, name, classmethod(log_call(obj.__func__)))
        elif inspect.isfunction(obj):
            setattr(cls, name, log_call(obj))
    return cls


@log_methods
class MathTools:
    def double(self, x):
        return 2 * x

    @staticmethod
    def square(x):
        return x * x

    @classmethod
    def identity(cls):
        return cls.__name__

In [16]:
m = MathTools()
assert m.double(4) == 8
assert MathTools.square(5) == 25
assert MathTools.identity() == "MathTools"
assert isinstance(vars(MathTools)["square"], staticmethod)
assert isinstance(vars(MathTools)["identity"], classmethod)
print("Descriptors preserved.")

CALL MathTools.double args=(<__main__.MathTools object at 0x0000023F2D0CEE40>, 4) kwargs={} -> 8
CALL MathTools.square args=(5,) kwargs={} -> 25
CALL MathTools.identity args=(<class '__main__.MathTools'>,) kwargs={} -> 'MathTools'
Descriptors preserved.


---
## Problem 9 — Decorate property getter, setter, and deleter

Extend class-wide logging to properties. Rebuild a new property from decorated `fget`, `fset`, and `fdel`, preserving the docstring.

In [17]:
def decorate_property(prop: property, fn_decorator: Callable) -> property:
    fget = fn_decorator(prop.fget) if prop.fget is not None else None
    fset = fn_decorator(prop.fset) if prop.fset is not None else None
    fdel = fn_decorator(prop.fdel) if prop.fdel is not None else None
    return property(fget=fget, fset=fset, fdel=fdel, doc=prop.__doc__)


def log_methods_and_properties(cls):
    for name, obj in list(vars(cls).items()):
        if isinstance(obj, staticmethod):
            setattr(cls, name, staticmethod(log_call(obj.__func__)))
        elif isinstance(obj, classmethod):
            setattr(cls, name, classmethod(log_call(obj.__func__)))
        elif isinstance(obj, property):
            setattr(cls, name, decorate_property(obj, log_call))
        elif inspect.isfunction(obj):
            setattr(cls, name, log_call(obj))
    return cls


@log_methods_and_properties
class Profile:
    def __init__(self, name):
        self._name = name

    @property
    def name(self):
        "The profile name."
        return self._name

    @name.setter
    def name(self, value):
        if not value:
            raise ValueError("name cannot be empty")
        self._name = value

    @name.deleter
    def name(self):
        del self._name

In [18]:
profile = Profile("Ada")
assert profile.name == "Ada"
profile.name = "Grace"
assert profile.name == "Grace"
assert vars(Profile)["name"].__doc__ == "The profile name."
del profile.name
assert not hasattr(profile, "_name")
print("Property getter/setter/deleter preserved and wrapped.")

CALL Profile.__init__ args=(<__main__.Profile object at 0x0000023F2D0CF380>, 'Ada') kwargs={} -> None
CALL Profile.name args=(<__main__.Profile object at 0x0000023F2D0CF380>,) kwargs={} -> 'Ada'
CALL Profile.name args=(<__main__.Profile object at 0x0000023F2D0CF380>, 'Grace') kwargs={} -> None
CALL Profile.name args=(<__main__.Profile object at 0x0000023F2D0CF380>,) kwargs={} -> 'Grace'
CALL Profile.name args=(<__main__.Profile object at 0x0000023F2D0CF380>,) kwargs={} -> None
Property getter/setter/deleter preserved and wrapped.


---
## Problem 10 — Avoid the `callable(...)` trap

Demonstrate why blindly wrapping every callable is unsafe: nested classes and callable instances are callables too. Then fix it by explicitly selecting plain functions.

In [19]:
def marker(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)
    wrapper.__marked__ = True
    return wrapper


def naive_wrap_callables(cls):
    for name, obj in list(vars(cls).items()):
        if callable(obj):
            setattr(cls, name, marker(obj))
    return cls


@naive_wrap_callables
class NaiveContainer:
    class Nested:
        pass

    helper = CallableHelper()

    def method(self):
        return "ok"


print("Naive Nested type:", type(NaiveContainer.Nested).__name__)
print("Naive helper type:", type(NaiveContainer.helper).__name__)

Naive Nested type: function
Naive helper type: function


In [20]:
def safe_wrap_functions(cls):
    for name, obj in list(vars(cls).items()):
        if inspect.isfunction(obj):
            setattr(cls, name, marker(obj))
    return cls


@safe_wrap_functions
class SafeContainer:
    class Nested:
        pass

    helper = CallableHelper()

    def method(self):
        return "ok"


assert inspect.isclass(SafeContainer.Nested)
assert isinstance(SafeContainer.helper, CallableHelper)
assert SafeContainer().method() == "ok"
print("Safe nested class and callable object preserved.")

Safe nested class and callable object preserved.


---
## Problem 11 — Configurable class-wide decorator

Build `decorate_members(...)` with options for private names, dunder names, and properties. Support ordinary, static, class, and property members while ignoring nested classes and arbitrary callable objects.

In [21]:
def decorate_members(
    fn_decorator: Callable[[Callable], Callable],
    *,
    include_private: bool = False,
    include_dunder: bool = False,
    include_properties: bool = True,
):
    def decorate_class(cls):
        for name, obj in list(vars(cls).items()):
            is_dunder = name.startswith("__") and name.endswith("__")
            is_private = name.startswith("_") and not is_dunder

            if is_dunder and not include_dunder:
                continue
            if is_private and not include_private:
                continue

            if isinstance(obj, staticmethod):
                setattr(cls, name, staticmethod(fn_decorator(obj.__func__)))
            elif isinstance(obj, classmethod):
                setattr(cls, name, classmethod(fn_decorator(obj.__func__)))
            elif include_properties and isinstance(obj, property):
                setattr(cls, name, decorate_property(obj, fn_decorator))
            elif inspect.isfunction(obj):
                setattr(cls, name, fn_decorator(obj))
        return cls
    return decorate_class


@decorate_members(log_call, include_private=False, include_dunder=False, include_properties=True)
class Inventory:
    def __init__(self):
        self._items = []

    def add(self, item):
        self._items.append(item)
        return len(self._items)

    def _internal_count(self):
        return len(self._items)

    @property
    def count(self):
        return len(self._items)

    @staticmethod
    def normalize(value):
        return str(value).strip().lower()

    @classmethod
    def category(cls):
        return cls.__name__

In [22]:
inv = Inventory()
assert inv.add("A") == 1
assert inv.count == 1
assert inv._internal_count() == 1
assert Inventory.normalize("  HELLO  ") == "hello"
assert Inventory.category() == "Inventory"
assert getattr(vars(Inventory)["add"], "__logged__", False)
assert not getattr(vars(Inventory)["_internal_count"], "__logged__", False)
print("Configurable filtering works.")

CALL Inventory.add args=(<__main__.Inventory object at 0x0000023F2D0CEF90>, 'A') kwargs={} -> 1
CALL Inventory.count args=(<__main__.Inventory object at 0x0000023F2D0CEF90>,) kwargs={} -> 1
CALL Inventory.normalize args=('  HELLO  ',) kwargs={} -> 'hello'
CALL Inventory.category args=(<class '__main__.Inventory'>,) kwargs={} -> 'Inventory'
Configurable filtering works.


---
## Problem 12 — Exclusion marker

Add `@do_not_decorate` so selected members are skipped by a class-wide decorator. The marker must work for ordinary, static, and class methods when placed on the underlying function.

In [23]:
def do_not_decorate(fn):
    fn.__skip_class_decorator__ = True
    return fn


def should_skip(fn):
    return bool(getattr(fn, "__skip_class_decorator__", False))


def decorate_members_with_skip(fn_decorator: Callable[[Callable], Callable]):
    def decorate_class(cls):
        for name, obj in list(vars(cls).items()):
            if isinstance(obj, staticmethod):
                fn = obj.__func__
                if not should_skip(fn):
                    setattr(cls, name, staticmethod(fn_decorator(fn)))
            elif isinstance(obj, classmethod):
                fn = obj.__func__
                if not should_skip(fn):
                    setattr(cls, name, classmethod(fn_decorator(fn)))
            elif inspect.isfunction(obj) and not should_skip(obj):
                setattr(cls, name, fn_decorator(obj))
        return cls
    return decorate_class


@decorate_members_with_skip(log_call)
class ApiClient:
    def request(self, path):
        return f"GET {path}"

    @do_not_decorate
    def health_check(self):
        return "ok"

    @staticmethod
    @do_not_decorate
    def version():
        return "1.0"

    @classmethod
    @do_not_decorate
    def service_name(cls):
        return cls.__name__

In [24]:
assert ApiClient().request("/users") == "GET /users"
assert ApiClient().health_check() == "ok"
assert ApiClient.version() == "1.0"
assert ApiClient.service_name() == "ApiClient"
assert getattr(vars(ApiClient)["request"], "__logged__", False)
assert not getattr(vars(ApiClient)["health_check"], "__logged__", False)
assert not getattr(vars(ApiClient)["version"].__func__, "__logged__", False)
assert not getattr(vars(ApiClient)["service_name"].__func__, "__logged__", False)
print("Opt-out marker works.")

CALL ApiClient.request args=(<__main__.ApiClient object at 0x0000023F2D0CF0E0>, '/users') kwargs={} -> 'GET /users'
Opt-out marker works.


---
## Problem 13 — Make decoration idempotent

Create `log_once(fn)` and `log_class_once(cls)` so applying the class decorator twice does not nest wrappers.

In [25]:
def log_once(fn):
    if getattr(fn, "__log_once_wrapped__", False):
        return fn

    @wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"ONCE {fn.__qualname__} -> {result!r}")
        return result

    wrapper.__log_once_wrapped__ = True
    return wrapper


def log_class_once(cls):
    for name, obj in list(vars(cls).items()):
        if isinstance(obj, staticmethod):
            setattr(cls, name, staticmethod(log_once(obj.__func__)))
        elif isinstance(obj, classmethod):
            setattr(cls, name, classmethod(log_once(obj.__func__)))
        elif inspect.isfunction(obj):
            setattr(cls, name, log_once(obj))
    return cls


@log_class_once
class Repeated:
    def ping(self):
        return "pong"


first = vars(Repeated)["ping"]
log_class_once(Repeated)
second = vars(Repeated)["ping"]

In [26]:
assert first is second
assert Repeated().ping() == "pong"
print("Same function object after second pass:", first is second)

ONCE Repeated.ping -> 'pong'
Same function object after second pass: True


---
## Problem 14 — Time methods even when exceptions occur

Create `timed(fn)` with `time.perf_counter()`. Timing must print in a `finally` block so an exception still records elapsed time and is re-raised unchanged.

In [27]:
def timed(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"TIMING {fn.__qualname__}: {elapsed:.6f}s")
    return wrapper


@decorate_members(timed, include_properties=False)
class TimedService:
    def work(self, n):
        return sum(range(n))

    def fail(self):
        raise RuntimeError("boom")

    @staticmethod
    def static_work(n):
        return n * 2

In [28]:
service = TimedService()
assert service.work(1000) == sum(range(1000))
assert TimedService.static_work(10) == 20
try:
    service.fail()
except RuntimeError as exc:
    assert str(exc) == "boom"
    print("Original exception preserved:", exc)

TIMING TimedService.work: 0.000024s
TIMING TimedService.static_work: 0.000002s
TIMING TimedService.fail: 0.000004s
Original exception preserved: boom


---
## Problem 15 — Validate instance state after `__init__`

Write `@validate_state`: after the original constructor runs, call `self.validate()`. Require a callable `validate` member and preserve constructor metadata.

In [29]:
def validate_state(cls):
    validate = getattr(cls, "validate", None)
    if not callable(validate):
        raise TypeError(f"{cls.__name__} must define validate(self)")

    original_init = cls.__init__

    @wraps(original_init)
    def wrapped_init(self, *args, **kwargs):
        original_init(self, *args, **kwargs)
        self.validate()

    cls.__init__ = wrapped_init
    return cls


@validate_state
class User:
    def __init__(self, username, age):
        self.username = username
        self.age = age

    def validate(self):
        if not self.username:
            raise ValueError("username is required")
        if self.age < 0:
            raise ValueError("age cannot be negative")

In [30]:
u = User("ada", 36)
assert (u.username, u.age) == ("ada", 36)
for args in [("", 10), ("bob", -1)]:
    try:
        User(*args)
    except ValueError as exc:
        print("Validation error:", exc)

Validation error: username is required
Validation error: age cannot be negative


---
## Problem 16 — Generate ordering methods from a key function

Implement `@ordered_by(key)`. Inject `__eq__` and `__lt__`, reject direct conflicts, return `NotImplemented` for different types, then apply `functools.total_ordering`.

In [31]:
def ordered_by(key: Callable[[Any], Any]):
    def decorate(cls):
        if "__eq__" in vars(cls) or "__lt__" in vars(cls):
            raise TypeError(f"{cls.__name__} must not define __eq__ or __lt__ directly")

        def __eq__(self, other):
            if type(self) is not type(other):
                return NotImplemented
            return key(self) == key(other)

        def __lt__(self, other):
            if type(self) is not type(other):
                return NotImplemented
            return key(self) < key(other)

        cls.__eq__ = __eq__
        cls.__lt__ = __lt__
        return total_ordering(cls)
    return decorate


@ordered_by(lambda obj: obj.score)
class Candidate:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    def __repr__(self):
        return f"Candidate({self.name!r}, {self.score})"

In [32]:
a = Candidate("A", 10)
b = Candidate("B", 20)
c = Candidate("C", 20)
assert a < b
assert b > a
assert b >= c
assert b == c
assert a != b
print(sorted([b, a, c]))

[Candidate('A', 10), Candidate('B', 20), Candidate('C', 20)]


---
## Problem 17 — Deliberate class replacement: cached instances

Build `@cached_instances(key=...)` by returning a wrapper subclass. This illustrates that a class decorator may intentionally return a different class, but that replacement should be explicit and metadata should be preserved.

In [33]:
def cached_instances(key: Callable[[tuple, dict], Any]):
    def decorate(cls):
        cache = {}

        class Cached(cls):
            def __new__(subcls, *args, **kwargs):
                cache_key = key(args, kwargs)
                if cache_key in cache:
                    return cache[cache_key]
                obj = super(Cached, subcls).__new__(subcls)
                cache[cache_key] = obj
                return obj

            def __init__(self, *args, **kwargs):
                cache_key = key(args, kwargs)
                if getattr(self, "__cached_init_key__", object()) == cache_key:
                    return
                super().__init__(*args, **kwargs)
                self.__cached_init_key__ = cache_key

        Cached.__name__ = cls.__name__
        Cached.__qualname__ = cls.__qualname__
        Cached.__module__ = cls.__module__
        Cached.__doc__ = cls.__doc__
        Cached.__wrapped__ = cls
        Cached.__instance_cache__ = cache
        return Cached
    return decorate


@cached_instances(key=lambda args, kwargs: (args, tuple(sorted(kwargs.items()))))
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

In [34]:
p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(2, 3)
assert p1 is p2
assert p1 is not p3
assert (p1.x, p1.y) == (1, 2)
print("Same key -> same instance:", p1 is p2)

Same key -> same instance: True


---
## Problem 18 — Per-class call statistics

Build `@count_class_calls` so instance, static, and class methods increment counters in `Class.__call_stats__`. Avoid global counters and late-binding closure mistakes.

In [35]:
def count_class_calls(cls):
    stats = {}
    cls.__call_stats__ = stats

    def make_counter(fn, member_name):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            stats[member_name] = stats.get(member_name, 0) + 1
            return fn(*args, **kwargs)
        return wrapper

    for name, obj in list(vars(cls).items()):
        if isinstance(obj, staticmethod):
            setattr(cls, name, staticmethod(make_counter(obj.__func__, name)))
        elif isinstance(obj, classmethod):
            setattr(cls, name, classmethod(make_counter(obj.__func__, name)))
        elif inspect.isfunction(obj):
            setattr(cls, name, make_counter(obj, name))
    return cls


@count_class_calls
class CounterService:
    def ping(self):
        return "pong"

    @staticmethod
    def double(x):
        return 2 * x

    @classmethod
    def name(cls):
        return cls.__name__

In [36]:
service = CounterService()
service.ping(); service.ping()
CounterService.double(5)
CounterService.name(); CounterService.name(); CounterService.name()
assert CounterService.__call_stats__["ping"] == 2
assert CounterService.__call_stats__["double"] == 1
assert CounterService.__call_stats__["name"] == 3
print(CounterService.__call_stats__)

{'ping': 2, 'double': 1, 'name': 3}


---
## Problem 19 — Inheritance: direct vs inherited members

Compare a decorator that transforms only methods directly in `vars(cls)` with one that also copies decorated inherited methods onto the subclass. Understand the semantic change before using the second approach.

In [37]:
def add_trace_label(label):
    def decorate(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            result = fn(*args, **kwargs)
            print(f"[{label}] {fn.__qualname__}")
            return result
        return wrapper
    return decorate


def decorate_direct_methods(label):
    fn_decorator = add_trace_label(label)
    def decorate(cls):
        for name, obj in list(vars(cls).items()):
            if inspect.isfunction(obj):
                setattr(cls, name, fn_decorator(obj))
        return cls
    return decorate


def decorate_visible_methods(label):
    fn_decorator = add_trace_label(label)
    def decorate(cls):
        names = {
            name for base in cls.__mro__ for name in vars(base)
            if not (name.startswith("__") and name.endswith("__"))
        }
        for name in names:
            raw = inspect.getattr_static(cls, name, None)
            if inspect.isfunction(raw):
                setattr(cls, name, fn_decorator(raw))
        return cls
    return decorate


class BaseGreeter:
    def inherited(self):
        return "base"


@decorate_direct_methods("DIRECT")
class DirectChild(BaseGreeter):
    def own(self):
        return "direct"


@decorate_visible_methods("VISIBLE")
class VisibleChild(BaseGreeter):
    def own(self):
        return "visible"

In [38]:
assert "inherited" not in vars(DirectChild)
assert "inherited" in vars(VisibleChild)
assert DirectChild().inherited() == "base"
assert VisibleChild().inherited() == "base"
print("Direct owns inherited?", "inherited" in vars(DirectChild))
print("Visible owns inherited?", "inherited" in vars(VisibleChild))

[VISIBLE] BaseGreeter.inherited
Direct owns inherited? False
Visible owns inherited? True


---
## Problem 20 — Production-style reusable member transformer

Create `transform_class_members` with support for ordinary methods, static/class methods, properties, name filtering, idempotence markers, and safe type selection. Then build `@instrumented` on top of it.

In [39]:
def transform_class_members(
    cls,
    *,
    fn_decorator: Callable[[Callable], Callable],
    name_predicate: Callable[[str], bool] = lambda name: True,
    marker: str | None = None,
    include_properties: bool = True,
):
    def transform_function(fn):
        if marker and getattr(fn, marker, False):
            return fn
        transformed = fn_decorator(fn)
        if marker:
            setattr(transformed, marker, True)
        return transformed

    for name, obj in list(vars(cls).items()):
        if not name_predicate(name):
            continue
        if isinstance(obj, staticmethod):
            setattr(cls, name, staticmethod(transform_function(obj.__func__)))
        elif isinstance(obj, classmethod):
            setattr(cls, name, classmethod(transform_function(obj.__func__)))
        elif include_properties and isinstance(obj, property):
            fget = transform_function(obj.fget) if obj.fget is not None else None
            fset = transform_function(obj.fset) if obj.fset is not None else None
            fdel = transform_function(obj.fdel) if obj.fdel is not None else None
            setattr(cls, name, property(fget, fset, fdel, obj.__doc__))
        elif inspect.isfunction(obj):
            setattr(cls, name, transform_function(obj))
    return cls


def instrumented(cls):
    return transform_class_members(
        cls,
        fn_decorator=log_call,
        name_predicate=lambda name: not name.startswith("_"),
        marker="__instrumented__",
        include_properties=True,
    )


@instrumented
class Order:
    def __init__(self, total):
        self._total = total

    def total_with_tax(self, rate):
        return self._total * (1 + rate)

    @property
    def total(self):
        "Pre-tax total."
        return self._total

    @staticmethod
    def currency():
        return "USD"

    @classmethod
    def type_name(cls):
        return cls.__name__

    def _internal(self):
        return "not instrumented"

In [40]:
order = Order(100)
assert order.total_with_tax(0.2) == 120
assert order.total == 100
assert Order.currency() == "USD"
assert Order.type_name() == "Order"
assert order._internal() == "not instrumented"
assert getattr(vars(Order)["total_with_tax"], "__instrumented__", False)
assert getattr(vars(Order)["currency"].__func__, "__instrumented__", False)
assert getattr(vars(Order)["type_name"].__func__, "__instrumented__", False)
assert getattr(vars(Order)["total"].fget, "__instrumented__", False)
assert not getattr(vars(Order)["_internal"], "__instrumented__", False)
before = vars(Order)["total_with_tax"]
instrumented(Order)
after = vars(Order)["total_with_tax"]
assert before is after
print("Reusable transformer passed all checks.")

CALL Order.total_with_tax args=(<__main__.Order object at 0x0000023F2D0CF4D0>, 0.2) kwargs={} -> 120.0
CALL Order.total args=(<__main__.Order object at 0x0000023F2D0CF4D0>,) kwargs={} -> 100
CALL Order.currency args=() kwargs={} -> 'USD'
CALL Order.type_name args=(<class '__main__.Order'>,) kwargs={} -> 'Order'
Reusable transformer passed all checks.


---
# Bonus debugging drills

## Drill A — Wrong decorator order

Why can this fail for a function decorator that expects a plain function?

```python
class Example:
    @log_call
    @staticmethod
    def ping():
        return "pong"
```

**Solution:** decorator application is bottom-up, so `log_call` receives the `staticmethod` descriptor. Prefer `@staticmethod` above `@log_call`, or let a class decorator unwrap and re-wrap the descriptor.

## Drill B — `getattr` vs `vars`

`getattr(cls, name)` performs descriptor binding. `vars(cls)[name]` or `inspect.getattr_static` lets a class transformer see the raw descriptor.

## Drill C — Stable namespace iteration

Using `list(vars(cls).items())` gives the transformer a stable snapshot while it calls `setattr` during the pass.

In [41]:
class InspectionLab:
    def method(self):
        pass

    @staticmethod
    def static():
        pass

    @classmethod
    def class_method(cls):
        pass


for name in ("method", "static", "class_method"):
    raw = vars(InspectionLab)[name]
    normal = getattr(InspectionLab, name)
    static = inspect.getattr_static(InspectionLab, name)
    print(
        name,
        "| vars:", type(raw).__name__,
        "| getattr:", type(normal).__name__,
        "| getattr_static:", type(static).__name__,
    )

method | vars: function | getattr: function | getattr_static: function
static | vars: staticmethod | getattr: function | getattr_static: staticmethod
class_method | vars: classmethod | getattr: method | getattr_static: classmethod


---
# Final challenge — Build `@audited`

Requirements:

1. Log all public instance, static, and class methods.
2. Log public property getters/setters/deleters.
3. Support an opt-out marker.
4. Preserve descriptors.
5. Do not modify nested classes or callable objects.
6. Be idempotent.
7. Store per-member call counts on the class.
8. Preserve function metadata.
9. Leave private and dunder members untouched.
10. Prove behavior with assertions.

A complete reference solution follows.

In [42]:
def audit_skip(fn):
    fn.__audit_skip__ = True
    return fn


def audited(cls):
    stats = getattr(cls, "__audit_stats__", None)
    if stats is None:
        stats = {}
        cls.__audit_stats__ = stats

    def decorate_function(fn, member_name):
        if getattr(fn, "__audit_skip__", False):
            return fn
        if getattr(fn, "__audited__", False):
            return fn

        @wraps(fn)
        def wrapper(*args, **kwargs):
            stats[member_name] = stats.get(member_name, 0) + 1
            try:
                result = fn(*args, **kwargs)
                print(f"AUDIT {fn.__qualname__} -> {result!r}")
                return result
            except Exception as exc:
                print(f"AUDIT {fn.__qualname__} raised {type(exc).__name__}: {exc}")
                raise

        wrapper.__audited__ = True
        return wrapper

    for name, obj in list(vars(cls).items()):
        if name.startswith("_"):
            continue
        if isinstance(obj, staticmethod):
            setattr(cls, name, staticmethod(decorate_function(obj.__func__, name)))
        elif isinstance(obj, classmethod):
            setattr(cls, name, classmethod(decorate_function(obj.__func__, name)))
        elif isinstance(obj, property):
            fget = decorate_function(obj.fget, name) if obj.fget is not None else None
            fset = decorate_function(obj.fset, name) if obj.fset is not None else None
            fdel = decorate_function(obj.fdel, name) if obj.fdel is not None else None
            setattr(cls, name, property(fget, fset, fdel, obj.__doc__))
        elif inspect.isfunction(obj):
            setattr(cls, name, decorate_function(obj, name))
    return cls


@audited
class Account:
    helper = CallableHelper()

    def __init__(self, balance):
        self._balance = balance

    def deposit(self, amount):
        self._balance += amount
        return self._balance

    @audit_skip
    def health(self):
        return "ok"

    @property
    def balance(self):
        "Current balance."
        return self._balance

    @balance.setter
    def balance(self, value):
        self._balance = value

    @staticmethod
    def institution():
        return "Example Bank"

    @classmethod
    def class_name(cls):
        return cls.__name__

    def _secret(self):
        return "private"

In [43]:
account = Account(100)
assert account.deposit(25) == 125
assert account.balance == 125
account.balance = 200
assert account.balance == 200
assert Account.institution() == "Example Bank"
assert Account.class_name() == "Account"
assert account.health() == "ok"
assert not getattr(vars(Account)["health"], "__audited__", False)
assert account._secret() == "private"
assert not getattr(vars(Account)["_secret"], "__audited__", False)
assert isinstance(Account.helper, CallableHelper)
assert vars(Account)["balance"].__doc__ == "Current balance."
before = vars(Account)["deposit"]
audited(Account)
after = vars(Account)["deposit"]
assert before is after
print("Audit stats:", Account.__audit_stats__)
print("Final challenge passed.")

AUDIT Account.deposit -> 125
AUDIT Account.balance -> 125
AUDIT Account.balance -> None
AUDIT Account.balance -> 200
AUDIT Account.institution -> 'Example Bank'
AUDIT Account.class_name -> 'Account'
Audit stats: {'deposit': 1, 'balance': 3, 'institution': 1, 'class_name': 1}
Final challenge passed.


---
# Review checklist

You should now be able to explain and implement:

- class mutation vs class replacement,
- parameterized class decorators,
- injected attributes and methods,
- registries and class validation,
- raw class namespaces and descriptor binding,
- `staticmethod.__func__` and `classmethod.__func__`,
- `property.fget`, `property.fset`, and `property.fdel`,
- rebuilding properties,
- `inspect.isfunction` and `inspect.getattr_static`,
- why generic `callable(...)` checks are often too broad,
- `functools.wraps`, opt-out markers, and idempotence,
- inheritance semantics,
- reusable class-member transformation,
- class-level instrumentation and statistics.

## Further extensions

Try adding async method support, exception filters, timing totals, configurable name patterns, inherited descriptor handling, or custom descriptors.